# 01. RAG Agent — Vector Search-Based Question Answering

## Learning Objectives

- Build a vector search pipeline with `InMemoryVectorStore`
- Define a retrieval tool with the `content_and_artifact` return pattern
- Create and query a RAG agent with `create_deep_agent`
- Apply v1 middleware (`ModelCallLimitMiddleware`, `ToolRetryMiddleware`)
- Use the **Skills system** to progressively disclose RAG domain knowledge


## Overview

| Item | Details |
|------|------|
| **Framework** | LangChain + Deep Agents |
| **Core components** | `InMemoryVectorStore`, `OpenAIEmbeddings`, `RecursiveCharacterTextSplitter` |
| **Agent pattern** | `content_and_artifact` tool → `create_deep_agent` |
| **Backend** | `FilesystemBackend(root_dir=".", virtual_mode=True)` |
| **Skill** | `skills/rag-agent/SKILL.md` — progressive disclosure of RAG domain knowledge |


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"


In [ ]:
# Optional observability setup
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## Step 1: Create Sample Documents

The first step in a RAG pipeline is preparing the documents you want to search. In a real system, you would load documents from PDFs, web pages, or databases. Here, for learning purposes, we create `Document` objects directly.


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM applications. It supports tools, chains, and agents.", metadata={"source": "langchain"}),
    Document(page_content="LangGraph is a framework for building stateful workflows. It provides a Graph API and a Functional API.", metadata={"source": "langgraph"}),
    Document(page_content="Deep Agents is an all-in-one agent SDK. It creates agents with create_deep_agent and supports backends and subagents.", metadata={"source": "deepagents"}),
    Document(page_content="RAG stands for retrieval-augmented generation. It injects external knowledge into an LLM to produce more accurate answers.", metadata={"source": "rag"}),
    Document(page_content="A vector store is a database that stores embeddings and performs similarity search. FAISS and Chroma are common examples.", metadata={"source": "vectorstore"}),
    Document(page_content="An agent is a system in which an LLM uses tools to perform tasks autonomously. ReAct is a representative pattern.", metadata={"source": "agent"}),
]
print(f"Created {len(docs)} documents")


## Step 2: Split the Text

Split larger documents into chunks that are easier to retrieve. `RecursiveCharacterTextSplitter` tries to split at natural boundaries such as paragraphs, sentences, and then words.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=50
)
splits = splitter.split_documents(docs)
print(f"Split result: {len(splits)} chunks")


## Step 3: Build the Vector Store

Convert the text into vectors with an OpenAI embedding model and store them in `InMemoryVectorStore`. In production, you would typically use a persistent store such as FAISS or Chroma.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)
print(f"Vector store ready — embedded {len(splits)} documents")


## Step 4: Define a Retrieval Tool (`content_and_artifact`)

The `response_format="content_and_artifact"` pattern makes the tool return two things:
- **content**: a text summary shown to the agent
- **artifact**: the full `Document` objects for downstream processing

This pattern helps save context tokens while still preserving access to the original data.


In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Search for relevant documents in the vector store."""
    results = vectorstore.similarity_search(query, k=3)
    content = "\n\n".join(d.page_content for d in results)
    return content, results


## Step 5: Test the Retrieval Tool by Itself

Before connecting the tool to the agent, verify that it behaves correctly.


In [ ]:
result = retrieve.invoke({"query": "What is an agent?"})
print(result)


## Step 6: Create the RAG Agent (with v1 Middleware)

Load the prompt from the prompt module. The flow tries LangSmith Hub → Langfuse → a local default prompt.

| Middleware | Role |
|---------|------|
| `ModelCallLimitMiddleware` | Prevents infinite loops by limiting the number of model calls |
| `ToolRetryMiddleware` | Automatically retries failed retrieval tool calls |


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolRetryMiddleware,
)
from prompts import RAG_AGENT_PROMPT

agent = create_deep_agent(
    model=model,
    tools=[retrieve],
    system_prompt=RAG_AGENT_PROMPT,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),
        ToolRetryMiddleware(max_retries=2),
    ],
)


## Step 7: Run a Simple Query and a Comparison Query

Use a simple query (single retrieval) and a comparison-style query (multiple retrieval steps) to verify that the RAG agent works as expected.


In [ ]:
# Simple query
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is LangGraph?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)


## Summary

| Item | Key Point |
|------|------|
| **Vector store** | `InMemoryVectorStore.from_documents()` — embedding-based similarity search |
| **Retrieval tool** | `@tool(response_format="content_and_artifact")` — summary + original artifact separation |
| **Agent** | `create_deep_agent(model, tools=[retrieve], backend=..., skills=["/skills/"])` |
| **Skill** | `skills/rag-agent/SKILL.md` — saves tokens through progressive disclosure |

---

**References:**
- `docs/langchain/24-retrieval.md`
- [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- `docs/deepagents/10-skills.md`

**Next Step:** → [02_sql_agent.ipynb](./02_sql_agent.ipynb): Build a SQL agent.
